In [ ]:
import pandas as pd
from pathlib import Path
import re

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
BASE_DIR = PROJECT_ROOT
RAW_DIR = BASE_DIR / "data" / "raw" / "core"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

lyrics_raw = pd.read_csv(RAW_DIR / "genius_lyrics_raw.csv")
tracks = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")

lyrics_raw.shape

In [ ]:
lyrics_raw["search_status"].value_counts()

In [ ]:
lyrics_raw.head()

In [ ]:
lyrics_clean = lyrics_raw[
    lyrics_raw["search_status"] == "exact"
].copy()

lyrics_clean.shape

In [ ]:
def clean_genius_lyrics(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # Remove common Genius embed/footer noise
    text = re.sub(r"\d*Embed$", "", text)
    text = re.sub(r"You might also like", "", text)
    
    # Remove excessive whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    
    return text.strip()

In [ ]:
lyrics_clean["lyrics_clean"] = lyrics_clean["lyrics"].apply(clean_genius_lyrics)

lyrics_clean[["track_name", "lyrics_clean"]].head()

In [ ]:
lyrics_clean["lyrics_clean"].isna().sum()

In [ ]:
lyrics_clean["lyrics_clean"].str.len().describe()

## Check for duplicate lyrics


In [ ]:
lyrics_clean["lyrics_clean"].duplicated().sum()

In [ ]:
duplicates = lyrics_clean[
    lyrics_clean["lyrics_clean"].duplicated(keep=False)
].sort_values("lyrics_clean")

duplicates[[
    "track_name",
    "album_name", 
    "genius_title"
]]

## Empty or suspicious lyrics

In [ ]:
lyrics_clean[
    lyrics_clean["lyrics_clean"].str.len() < 100
][["track_name", "lyrics_clean"]]

## Check for Genius artifacts

Sometimes Genius leaves things like:

- Embed
- 123Embed
- Contributors
- Lyrics

In [ ]:
patterns = [
    "embed", 
    "Contributors",
    "You might also like"
]

for p in patterns: 
    print(p, lyrics_clean["lyrics_clean"].str.contains(p, case=False).sum())

In [ ]:
lyrics_clean.sample(5)[
    ["track_name","lyrics_clean"]
]

## Lyrics statistics

In [ ]:
lyrics_clean["character_count"] = lyrics_clean["lyrics_clean"].str.len()

lyrics_clean["word_count"] = (
    lyrics_clean["lyrics_clean"]
    .str.split()
    .str.len()
)

In [ ]:
lyrics_clean[
    ["character_count", "word_count"]
].describe()

In [ ]:
lyrics_clean.to_csv(
    PROCESSED_DIR / "song_lyrics_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "song_lyrics_clean.csv")
print("Shape:", lyrics_clean.shape)

In [ ]:
lyrics_check = pd.read_csv(
    PROCESSED_DIR / "song_lyrics_clean.csv"
)

print(lyrics_check.shape)
lyrics_check.head()